## C-Drama RAG: Generator Prompt Lab

A scratchpad for iterating on the generator prompt in `pipeline.py`.
Hallucination resistance and off-topic handling are tested in the eval suite —
this notebook is purely about **prompt shape**: how phrasing, format, and
context size affect output quality and cost.

| What | Where |
|---|---|
| Generator prompt wording / format | **this notebook** |
| Token cost & context-size tradeoffs | **this notebook** |
| Synopsis length vs output quality | **this notebook** |
| Temperature & format stability | **this notebook** |
| Hallucination / guardrail tests | `tests/evals/` |
| Parser accuracy | `04a_parser_lab.ipynb` |

Prompt under test: `RECOMMEND_SYSTEM_PROMPT` in `pipeline.py` → model `gpt-4o-mini`

---
## Setup

In [ ]:
import os
import textwrap
from dotenv import load_dotenv
from openai import OpenAI

# Import the live production prompt and context builder so this notebook always
# tests what's actually running in pipeline.py — no copy-paste drift.
from src.recommender.pipeline import RECOMMEND_SYSTEM_PROMPT, build_context
from src.env import load_secrets

load_secrets()
client = OpenAI()

MODEL = "gpt-4o-mini"  # cheap, fast — right model for this task

def token_cost(usage, input_price=0.15, output_price=0.60) -> str:
    """Rough cost estimate in USD. gpt-4o-mini pricing per 1M tokens."""
    input_cost  = (usage.prompt_tokens     / 1_000_000) * input_price
    output_cost = (usage.completion_tokens / 1_000_000) * output_price
    total = input_cost + output_cost
    return (
        f"in={usage.prompt_tokens} tok  "
        f"out={usage.completion_tokens} tok  "
        f"≈ ${total:.6f}"
    )

print("Ready.")


---
## Mock candidates

Three candidate sets — one per search mode — so we can test the generator
against realistic input shapes without a DB connection.

| Mode | Similarity values | When it occurs |
|---|---|---|
| **reference** | High (0.6–0.9) | "Something like Nirvana in Fire" |
| **semantic** | Moderate (0.6–0.75) | "A drama where a woman fights for power" |
| **sql** | 0.0 everywhere | "Historical, rated above 8" |

In [ ]:
# --- Reference mode mock candidates (high similarity from anchor drama) ---
MOCK_DRAMAS_REFERENCE = [
    {
        "title": "Nirvana in Fire",
        "year": 2015,
        "mdl_score": 9.1,
        "similarity": 0.91,
        "ensemble_score": 0.88,
        "genres": ["Historical", "Mystery", "Political"],
        "tags": ["Adapted From A Novel", "Political Intrigue", "Revenge", "Strategic Male Lead", "Strong Male Lead"],
        "synopsis": "Mei Changsu, a brilliant strategist, returns to the capital under a new identity to clear his father's name and avenge the injustice done to his family, while navigating treacherous court politics."
    },
    {
        "title": "Story of Ming Lan",
        "year": 2018,
        "mdl_score": 8.7,
        "similarity": 0.84,
        "ensemble_score": 0.81,
        "genres": ["Historical", "Romance", "Family"],
        "tags": ["Smart Female Lead", "Scheming", "Adapted From A Novel", "Aristocracy", "Strong Male Lead"],
        "synopsis": "Ming Lan, a low-ranked concubine's daughter, uses her wits to survive in an aristocratic household, slowly earning her place while hiding her true intelligence — until the right moment to act."
    },
    {
        "title": "The Long Ballad",
        "year": 2021,
        "mdl_score": 8.2,
        "similarity": 0.78,
        "ensemble_score": 0.76,
        "genres": ["Historical", "Romance", "Action"],
        "tags": ["Strong Female Lead", "Tang Dynasty", "Adapted From A Manga", "Enemies to Lovers", "War"],
        "synopsis": "A Tang dynasty princess escapes a palace massacre and joins a resistance movement, falling in love with a Göktürk warrior while fighting to reclaim her family's honor."
    },
    {
        "title": "Love Between Fairy and Devil",
        "year": 2022,
        "mdl_score": 8.5,
        "similarity": 0.72,
        "ensemble_score": 0.72,
        "genres": ["Fantasy", "Romance", "Wuxia"],
        "tags": ["Body Swap", "Immortals", "Enemies to Lovers", "Strong Male Lead", "Adapted From A Novel"],
        "synopsis": "A lowly fairy accidentally frees a powerful devil king from his 30,000-year prison. Their souls become linked, forcing them together on a journey that blurs the line between hatred and love."
    },
    {
        "title": "Hikaru no Go (C-Drama)",
        "year": 2020,
        "mdl_score": 7.8,
        "similarity": 0.61,
        "ensemble_score": 0.64,
        "genres": ["Youth", "Sports"],
        "tags": ["Board Games", "Coming of Age"],
        "synopsis": "A young boy discovers a Go board haunted by the spirit of an ancient master and learns the game under his guidance."
    },
]

# --- Semantic mode mock candidates (moderate similarity from query embedding) ---
MOCK_DRAMAS_SEMANTIC = [
    {
        "title": "Who Rules the World",
        "year": 2022,
        "mdl_score": 7.9,
        "similarity": 0.74,
        "ensemble_score": 0.70,
        "genres": ["Historical", "Romance", "Action"],
        "tags": ["Strong Female Lead", "Strong Male Lead", "Martial Arts", "Power Couple"],
        "synopsis": "Two martial arts prodigies — a woman with unmatched intellect and a fearless warrior — compete and fall in love while trying to save their kingdom from corrupt officials."
    },
    {
        "title": "The Sword and the Brocade",
        "year": 2021,
        "mdl_score": 7.6,
        "similarity": 0.68,
        "ensemble_score": 0.65,
        "genres": ["Historical", "Romance", "Family"],
        "tags": ["Smart Female Lead", "Scheming", "Aristocracy", "Adapted From A Novel"],
        "synopsis": "After being reborn, a young woman marries into a powerful military family and uses her knowledge of the future to protect her new household from scheming relatives."
    },
    {
        "title": "Ashes of Love",
        "year": 2018,
        "mdl_score": 8.6,
        "similarity": 0.65,
        "ensemble_score": 0.68,
        "genres": ["Fantasy", "Romance", "Wuxia"],
        "tags": ["Immortals", "Tragic Romance", "Adapted From A Novel", "Love Triangle"],
        "synopsis": "A flower deity, stripped of her ability to feel love by her mother's spell, becomes entangled with two immortal princes in a celestial war that spans lifetimes."
    },
]

# --- SQL mode mock candidates (similarity=0, ranked by quality + popularity) ---
MOCK_DRAMAS_SQL = [
    {
        "title": "Joy of Life",
        "year": 2019,
        "mdl_score": 9.0,
        "similarity": 0.0,
        "ensemble_score": 0.0,
        "watchers": 720_000,
        "genres": ["Historical", "Comedy", "Political"],
        "tags": ["Adapted From A Novel", "Smart Male Lead", "Time Travel", "Political Intrigue"],
        "synopsis": "A modern man reborn in an ancient world navigates palace politics with future knowledge, wit, and an irreverent attitude."
    },
    {
        "title": "The Story of Ming Lan",
        "year": 2018,
        "mdl_score": 8.7,
        "similarity": 0.0,
        "ensemble_score": 0.0,
        "watchers": 620_000,
        "genres": ["Historical", "Romance", "Family"],
        "tags": ["Smart Female Lead", "Scheming", "Aristocracy"],
        "synopsis": "Ming Lan, a low-ranked concubine's daughter, uses her wits to survive in an aristocratic household."
    },
    {
        "title": "The Rebel Princess",
        "year": 2021,
        "mdl_score": 8.1,
        "similarity": 0.0,
        "ensemble_score": 0.0,
        "watchers": 350_000,
        "genres": ["Historical", "Romance", "Political"],
        "tags": ["Strong Female Lead", "Political Intrigue", "War"],
        "synopsis": "A princess and a general form a political marriage that evolves into genuine love as they navigate power struggles in a crumbling dynasty."
    },
]

# Sanity-check: show what the generator actually sees for one reference drama
print("=== Sample context (reference mode, 1 drama) ===")
print(build_context(MOCK_DRAMAS_REFERENCE[:1]))
print()
print("=== Sample context (SQL mode, similarity=0) ===")
print(build_context(MOCK_DRAMAS_SQL[:1]))


---
## Part 1: Prompt variants

Three system prompt formats tested against all three candidate shapes.
The format × mode matrix matters: a paragraph prompt that works well for
reference mode (high similarity) should also work for SQL mode (similarity=0
everywhere, no semantic signal).

| Format | Output style | Token budget |
|---|---|---|
| Paragraphs (current) | Warm, conversational | ~300–500 out |
| JSON | `[{title, year, score, reason}]` | ~150–250 out |
| Bullets | `**Title (Year)** — one sentence` | ~100–200 out |

In [ ]:
# Format A — production paragraphs prompt (imported from pipeline.py)
RECOMMEND_PARAGRAPH = RECOMMEND_SYSTEM_PROMPT

# Format B — JSON structured output
RECOMMEND_JSON = """\
You are a Chinese drama recommender.
Recommend ONLY dramas from the provided context — never invent titles.
Pick the 3 best matches and return ONLY a JSON array, no other text:
[
  {"title": "...", "year": ..., "mdl_score": ..., "reason": "one sentence why it fits"}
]
reason must be specific to the user's request, under 25 words.\
"""

# Format C — compact bullets
RECOMMEND_BULLETS = """\
You are a Chinese drama recommender.
Recommend ONLY dramas from the provided context — never invent titles.
Pick the 3 best matches. For each, reply with:
**Title (Year)** — MDL: X.X — one sentence why it fits the request.
No other text before or after.\
"""

GENERATOR_FORMATS = {
    "paragraphs": (RECOMMEND_PARAGRAPH, 500),
    "json":       (RECOMMEND_JSON,       200),
    "bullets":    (RECOMMEND_BULLETS,    200),
}

MODE_TEST_CASES = {
    "reference": {
        "query": "I want something with political intrigue and a clever male lead, historical setting",
        "dramas": MOCK_DRAMAS_REFERENCE,
    },
    "semantic": {
        "query": "A drama where a woman disguises herself to survive in a dangerous world and slowly gains power",
        "dramas": MOCK_DRAMAS_SEMANTIC,
    },
    "sql": {
        "query": "Historical dramas rated above 8",
        "dramas": MOCK_DRAMAS_SQL,
    },
}

for mode_name, mode_cfg in MODE_TEST_CASES.items():
    print(f"\n{'#'*60}")
    print(f"# SEARCH MODE: {mode_name.upper()}")
    print(f"# Query: {mode_cfg['query']}")
    print(f"{'#'*60}")

    context = build_context(mode_cfg["dramas"])

    for fmt_name, (system_prompt, max_tok) in GENERATOR_FORMATS.items():
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=max_tok,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": f"My request: {mode_cfg['query']}\n\nDramas to choose from:\n{context}"},
            ],
        )
        output = response.choices[0].message.content
        cost   = token_cost(response.usage)

        print(f"\n{'='*60}")
        print(f"FORMAT: {fmt_name.upper()}   |   {cost}")
        print(f"{'='*60}")
        print(output)


---
## Custom prompt sandbox

Write your own system prompt variant here. Swap `SANDBOX_DRAMAS` to test it
against different candidate shapes. Cost is printed so you can compare directly
against the formats above.

In [ ]:
# --- Your prompt here ---
MY_PROMPT = """\
You are a Chinese drama recommender.
Recommend ONLY dramas from the provided context — never invent titles.
Pick the 3 best matches.

[YOUR INSTRUCTIONS HERE]
"""

SANDBOX_QUERY  = "I want something with political intrigue and a clever protagonist"
SANDBOX_DRAMAS = MOCK_DRAMAS_REFERENCE   # swap: MOCK_DRAMAS_SEMANTIC or MOCK_DRAMAS_SQL

context  = build_context(SANDBOX_DRAMAS)
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=500,
    messages=[
        {"role": "system", "content": MY_PROMPT},
        {"role": "user",   "content": f"My request: {SANDBOX_QUERY}\n\nDramas to choose from:\n{context}"},
    ],
)
print(f"Cost: {token_cost(response.usage)}")
print()
print(response.choices[0].message.content)


---
## Part 2: Token budget

How many tokens does each part of the prompt use?
Per-drama token cost lets you estimate call cost before running — useful for
deciding how many candidates to pass to the generator.

`gpt-4o-mini` pricing (Apr 2026): input $0.15 / 1M, output $0.60 / 1M tokens.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
TEST_QUERY = "I want something with political intrigue and a clever protagonist"

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print("=== Per-drama token cost (reference candidates) ===")
for drama in MOCK_DRAMAS_REFERENCE:
    ctx  = build_context([drama])
    toks = count_tokens(ctx)
    print(f"  {drama['title']:<40} {toks:>4} tokens")

print()
print("=== System prompt sizes ===")
print(f"  Paragraphs:  {count_tokens(RECOMMEND_PARAGRAPH):>4} tokens")
print(f"  JSON:        {count_tokens(RECOMMEND_JSON):>4} tokens")
print(f"  Bullets:     {count_tokens(RECOMMEND_BULLETS):>4} tokens")

print()
print("=== Projected cost per call (paragraphs prompt, ~400 output tokens) ===")
avg_drama_tokens = (
    sum(count_tokens(build_context([d])) for d in MOCK_DRAMAS_REFERENCE)
    // len(MOCK_DRAMAS_REFERENCE)
)
base_tokens = count_tokens(RECOMMEND_PARAGRAPH) + count_tokens(TEST_QUERY) + 20
est_out = 400

print(f"  {'N candidates':<14} {'Input tokens':<16} {'Est. cost (USD)'}")
print("  " + "-" * 46)
for n in [3, 5, 10]:
    total_in = base_tokens + avg_drama_tokens * n
    cost_usd = (total_in / 1_000_000 * 0.15) + (est_out / 1_000_000 * 0.60)
    print(f"  {n:<14} {total_in:<16} ${cost_usd:.6f}")


---
## Part 3: Synopsis truncation

Synopses dominate input token count. Truncating them cuts cost — but
how much output quality do you lose?

Run the same query at four truncation lengths and compare. Spoiler: at
100–200 chars the model often loses enough specific detail that
recommendations become generic; 400 chars is usually the sweet spot.

In [ ]:
TRUNC_QUERY  = "Something with political intrigue and a clever protagonist"
TRUNC_DRAMAS = MOCK_DRAMAS_REFERENCE[:3]

def build_context_truncated(dramas: list[dict], max_chars: int | None) -> str:
    parts = []
    for d in dramas:
        synopsis = d["synopsis"]
        if max_chars:
            synopsis = synopsis[:max_chars]
        parts.append(
            f"Title: {d['title']} ({d['year']}) | MDL Score: {d['mdl_score']}\n"
            f"Genres: {', '.join(d.get('genres') or [])}\n"
            f"Tags: {', '.join((d.get('tags') or [])[:5])}\n"
            f"Synopsis: {synopsis}"
        )
    return "\n\n".join(parts)

for max_chars in [100, 200, 400, None]:
    label = f"{max_chars} chars" if max_chars else "full"
    ctx   = build_context_truncated(TRUNC_DRAMAS, max_chars)
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=400,
        messages=[
            {"role": "system", "content": RECOMMEND_PARAGRAPH},
            {"role": "user",   "content": f"My request: {TRUNC_QUERY}\n\nDramas to choose from:\n{ctx}"},
        ],
    )
    print(f"{'='*60}")
    print(f"Synopsis: {label:<12}  |  {token_cost(response.usage)}")
    print(f"{'='*60}")
    print(response.choices[0].message.content)
    print()


---
## Part 4: Temperature & consistency

The generator runs at OpenAI's default temperature. At `temperature=0` output
should be nearly deterministic. Higher temperatures introduce variation — useful
for creative diversity but risky for structured formats (JSON breaks first).

Tests:
- Does `temperature=0` produce identical outputs across N runs?
- At `temperature=0.7`, do recommendations drift to different dramas?
- At `temperature=1.2`, does the JSON format break?

In [ ]:
CONSISTENCY_QUERY   = "Historical drama with a clever protagonist and political scheming"
CONSISTENCY_CONTEXT = build_context(MOCK_DRAMAS_REFERENCE[:3])
N_RUNS = 3

for temp in [0.0, 0.7, 1.2]:
    print(f"{'='*60}")
    print(f"Temperature = {temp}")
    outputs = []
    for i in range(N_RUNS):
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=temp,
            max_tokens=350,
            messages=[
                {"role": "system", "content": RECOMMEND_PARAGRAPH},
                {"role": "user",   "content": f"My request: {CONSISTENCY_QUERY}\n\nDramas to choose from:\n{CONSISTENCY_CONTEXT}"},
            ],
        )
        outputs.append(resp.choices[0].message.content.strip())

    first_picks = []
    for out in outputs:
        for drama in MOCK_DRAMAS_REFERENCE:
            if drama["title"].lower() in out.lower():
                first_picks.append(drama["title"])
                break
        else:
            first_picks.append("(none detected)")

    consistent = len(set(first_picks)) == 1
    print(f"  First drama mentioned across {N_RUNS} runs: {first_picks}")
    print(f"  Consistent first pick: {'yes' if consistent else 'no — drifts!'}")

    if temp == 0.0:
        identical = all(o == outputs[0] for o in outputs)
        print(f"  Outputs byte-identical: {'yes' if identical else 'no (minor float variance expected)'}")

    if temp == 1.2:
        json_resp = client.chat.completions.create(
            model=MODEL, temperature=temp, max_tokens=250,
            messages=[
                {"role": "system", "content": RECOMMEND_JSON},
                {"role": "user",   "content": f"My request: {CONSISTENCY_QUERY}\n\nDramas to choose from:\n{CONSISTENCY_CONTEXT}"},
            ],
        )
        raw = json_resp.choices[0].message.content.strip()
        try:
            import json as _json
            parsed = _json.loads(raw)
            print(f"  JSON format at temp=1.2: valid ({len(parsed)} items)")
        except _json.JSONDecodeError as e:
            print(f"  JSON format at temp=1.2: BROKEN — {e}")
            print(f"  Raw output: {raw[:200]}")

    print(f"\n  Sample output (run 1):\n")
    print(textwrap.indent(outputs[0][:400], "    "))
    print()


---
## Notes & findings

### Part 1: Prompt variants x mode (3 x 3 = 9 calls)
- All 9 ran cleanly -- no format breaks, no hallucinations across reference / semantic / sql modes.
- **Paragraphs**: warm, picks all 3 candidates, ~330-390 output tokens. Slight tendency to over-justify ("Although it centers on a female lead, the male character is...") when the candidate doesn't perfectly match -- graceful, but verbose.
- **JSON**: tightest output (~100-140 tokens). One quirk: in semantic mode it returned only 2 items (asked for 3) -- the third candidate's similarity was likely too low for the model to feel justified including it. Paragraphs and bullets always returned 3.
- **Bullets**: most consistent shape, ~120-150 tokens, format never wavered across runs.
- **SQL mode (all similarity=0)**: generator stayed grounded -- picked Joy of Life / Story of Ming Lan / Rebel Princess and reasoned from the synopsis text, not the (zeroed) similarity score. So the paragraphs prompt does **not** need a per-mode SQL variant.
- **Semantic mode**: reasons were anchored in the user's "disguise / gain power" phrasing -- the generator does ground recommendations in the description rather than just paraphrasing the candidate.

### Part 2: Token budget
- Per-drama input cost (full synopsis): **65-97 tokens**, avg ~86. Synopsis is the dominant component.
- System prompt sizes: paragraphs 103, JSON 78, bullets 60 tokens.
- Cost per call (paragraphs prompt, ~400 output tokens):

| Candidates | Input toks | Cost (USD) |
|---|---|---|
| 3 | 388 | $0.000298 |
| 5 | 558 | $0.000324 |
| 10 | 983 | $0.000387 |

- Going from 3 -> 10 candidates only adds ~30% to per-call cost (output tokens dominate at this prompt size). 10-candidate context is fine to keep -- no budgetary reason to trim further.

### Part 3: Synopsis truncation
- 100 chars: 375 out tokens, output stayed grounded but model **embellished from world knowledge** ("high-stakes political maneuvering", "deep desire for justice") -- phrases that aren't in the truncated synopsis. Not visibly wrong here because the dramas are well-known to the model, but unsafe for less-famous candidates.
- 200 chars: 337 out tokens, quality matched full.
- 400 chars: 389 out tokens, indistinguishable from full.
- Full: 335 out tokens, baseline.
- Input savings 100->full at 3 candidates: ~47 tokens. At 10 candidates: ~150 tokens. **The savings aren't large enough to justify shorter truncation given the embellishment risk.**
- **400 chars is the safe production setting.** It captures the full plot beat for nearly every drama in this dataset without leaving the model to invent.

### Part 4: Temperature & consistency
- temp=0.0: same first pick (Nirvana in Fire) on 3/3 runs, but outputs **not byte-identical** -- minor float variance is normal for OpenAI even at temp=0.
- temp=0.7: same first pick 3/3. The candidate gap (9.1 vs the next's 8.7) was wide enough that the obvious winner stayed obvious.
- temp=1.2: same first pick 3/3, **JSON format survived** (3 valid items, structurally clean).
- For this candidate set, temperature was not a quality risk in the tested range. **With borderline candidates** (3 dramas at ~8.5 each), drift could appear at 0.7+ -- worth keeping `temperature=0` in production.

### Decisions made
- `max_synopsis_chars`: keep at full (or set to **400** if a future ingestion adds longer synopses) -- the quality cliff is below 200 chars, savings above 400 are negligible.
- Keep paragraphs as the production format -- better UX for the chat product; JSON only worth it if the consumer is structured (e.g. UI cards, not chat).
- No per-mode prompt needed -- the paragraphs prompt handles all three search modes correctly.
- Keep `temperature=0` for the generator -- same as today.
- No code changes needed in `pipeline.py` for the generator. Parser changes (variant B) are the actionable item from this lab session -- see `04a_parser_lab.ipynb` Notes.
